In [ ]:
!unzip -q "/content/drive/MyDrive/clean_split.zip"

In [ ]:
import os
import io
import time
import copy
import json
import random
import logging
from collections import Counter

import cv2
import numpy as np
from PIL import Image, ImageChops, ImageEnhance, ImageFile
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, precision_score, recall_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models

# Allow loading of truncated/corrupted images commonly found in scraped datasets
ImageFile.LOAD_TRUNCATED_IMAGES = True

class Config:
    """Hyperparameters and configuration settings for SOTA Training."""
    # Data Paths
    TRAIN_RGB_DIR = "clean_split/train/rgb"
    TRAIN_NOISE_DIR = "clean_split/train/noise"  # If pre-computed, otherwise we generate on the fly
    VAL_RGB_DIR = "clean_split/val/rgb"
    CHECKPOINT_DIR = "checkpoints"
    BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "sota_best_fusion_model.pth")

    # SOTA Hyperparameters
    BATCH_SIZE = 32
    EPOCHS_PHASE_1 = 5      # Warm-up the fusion head
    EPOCHS_PHASE_2 = 25     # Fine-tune the backbones
    LR_FUSION = 1e-3
    LR_FINETUNE = 1e-5
    WEIGHT_DECAY = 1e-4
    PATIENCE = 5
    IMG_SIZE = 224

    # Hardware
    NUM_WORKERS = 2
    PIN_MEMORY = True
    SEED = 42

    # ImageNet Standards
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]

def set_seed(seed=42):
    """Ensures absolute reproducibility across experimental runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

def apply_clahe(pil_img):
    """Enhances hidden deepfake blending seams before ELA extraction."""
    img_cv = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(img_cv)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB))

def generate_ela(img, quality=90):
    """Calculates Error Level Analysis dynamically in RAM."""
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    compressed = Image.open(buffer)

    ela = ImageChops.difference(img, compressed)
    extrema = ela.getextrema()
    max_diff = max([ex[1] for ex in extrema]) if extrema else 1
    scale = min(255.0 / max(max_diff, 1), 50.0)
    return ImageEnhance.Brightness(ela).enhance(scale)

class SOTADualStreamDataset(Dataset):
    """
    Advanced Dataset that applies aggressive augmentations to the RGB stream
    but strictly preserves the forensic integrity of the Noise stream.
    """
    def __init__(self, rgb_dir, is_training=True):
        self.samples = []
        self.targets = []
        self.is_training = is_training

        # Load valid files
        for label, folder_name in [(1, "real"), (0, "fake")]:
            folder_path = os.path.join(rgb_dir, folder_name)
            if not os.path.exists(folder_path): continue

            for filename in os.listdir(folder_path):
                if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append(os.path.join(folder_path, filename))
                    self.targets.append(label)

        # 1. Shared Geometric Transforms (Safe for both streams)
        self.geo_transform = transforms.Compose([
            transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        ])

        # 2. RGB-Only Aggressive Transforms (Prevents Overfitting)
        self.rgb_aug = transforms.Compose([
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
        ])

        # 3. Final Tensor Normalization
        self.to_tensor_norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(Config.MEAN, Config.STD)
        ])

        # Noise Stream uses [0.5, 0.5, 0.5] norm since it's a difference map
        self.noise_norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        label = self.targets[idx]

        try:
            base_img = Image.open(path).convert('RGB')
        except Exception:
            # Fallback for corrupted image files during high-speed loading
            base_img = Image.new('RGB', (Config.IMG_SIZE, Config.IMG_SIZE), (0, 0, 0))

        # Apply shared geometry
        base_img = self.geo_transform(base_img)

        # Apply random horizontal flip manually to ensure both streams flip together
        if self.is_training and random.random() > 0.5:
            # FIX: Use Image.Transpose.FLIP_LEFT_RIGHT to avoid Pillow 10+ deprecation errors
            base_img = base_img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

        # Generate Noise Stream (MUST happen before RGB color augmentations)
        clahe_img = apply_clahe(base_img)
        ela_img = generate_ela(clahe_img, quality=90)
        noise_tensor = self.noise_norm(ela_img)

        # Generate RGB Stream (Apply aggressive color/pixel augmentations)
        rgb_tensor = transforms.ToTensor()(base_img)
        if self.is_training:
            rgb_tensor = self.rgb_aug(rgb_tensor)
        rgb_tensor = transforms.Normalize(Config.MEAN, Config.STD)(rgb_tensor)

        return rgb_tensor, noise_tensor, torch.tensor([label], dtype=torch.float32)

def get_dataloaders():
    """Builds loaders and applies WeightedRandomSampler for class imbalance."""
    train_ds = SOTADualStreamDataset(Config.TRAIN_RGB_DIR, is_training=True)
    val_ds = SOTADualStreamDataset(Config.VAL_RGB_DIR, is_training=False)

    # Calculate dataset weights to force 50/50 Real/Fake batches
    class_counts = Counter(train_ds.targets)
    logger.info(f"Training Class Counts: Real(1)={class_counts[1]}, Fake(0)={class_counts[0]}")

    weights = {0: 1.0 / class_counts[0], 1: 1.0 / class_counts[1]}
    sample_weights = torch.tensor([weights[t] for t in train_ds.targets])

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_ds, batch_size=Config.BATCH_SIZE, sampler=sampler,
        num_workers=Config.NUM_WORKERS, pin_memory=Config.PIN_MEMORY, drop_last=True
    )

    val_loader = DataLoader(
        val_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
        num_workers=Config.NUM_WORKERS, pin_memory=Config.PIN_MEMORY, drop_last=False
    )

    return train_loader, val_loader

class FusionHead(nn.Module):
    def __init__(self, in_features=1792):
        super().__init__()
        # Deep Fusion MLP with strong regularization
        self.net = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(p=0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, rgb_f, noise_f):
        return self.net(torch.cat([rgb_f, noise_f], dim=1))

class DualStreamDetector(nn.Module):
    def __init__(self):
        super().__init__()
        # Stream 1: RGB Spatial Features (EfficientNet-B0)
        self.stream1 = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.stream1.classifier = nn.Identity() # Outputs 1280

        # Stream 2: Noise Forensic Features (MobileNetV2)
        self.stream2 = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.stream2.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True)
        ) # Outputs 512

        self.fusion = FusionHead(in_features=1280 + 512)

    def forward(self, rgb, noise):
        return self.fusion(self.stream1(rgb), self.stream2(noise))

def calculate_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.0 # Handle cases where batch might only have 1 class during testing

    return {
        'acc': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc': auc,
        'cm': confusion_matrix(y_true, y_pred)
    }

def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, desc="Training", leave=False)
    for rgb, noise, labels in pbar:
        rgb, noise, labels = rgb.to(device, non_blocking=True), noise.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # FIX: Use standard torch.autocast to avoid PyTorch versioning API changes
        with torch.autocast(device_type=device.type):
            logits = model(rgb, noise)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    return running_loss / len(loader)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_targets, all_probs = [], []

    with torch.no_grad():
        for rgb, noise, labels in tqdm(loader, desc="Validating", leave=False):
            rgb, noise, labels = rgb.to(device, non_blocking=True), noise.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            # FIX: Use standard torch.autocast
            with torch.autocast(device_type=device.type):
                logits = model(rgb, noise)
                loss = criterion(logits, labels)

            running_loss += loss.item()
            probs = torch.sigmoid(logits)

            all_targets.extend(labels.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())

    metrics = calculate_metrics(np.array(all_targets), np.array(all_probs))
    return running_loss / len(loader), metrics

def main():
    set_seed(Config.SEED)
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"🚀 Initializing SOTA Training on {device}")

    train_loader, val_loader = get_dataloaders()
    model = DualStreamDetector().to(device)
    criterion = nn.BCEWithLogitsLoss()

    # FIX: Ensure GradScaler works across all PyTorch versions safely
    try:
        scaler = torch.amp.GradScaler(device.type)
    except TypeError:
        # Fallback for older PyTorch versions
        scaler = torch.cuda.amp.GradScaler()

    best_auc = 0.0
    patience_counter = 0

    # =========================================================================
    # PHASE 1: Warm-up Fusion Head (Freeze Backbones)
    # Prevents catastrophic forgetting of ImageNet features
    # =========================================================================
    logger.info("🔥 PHASE 1: Warming up Fusion Head...")
    for param in model.stream1.parameters(): param.requires_grad = False
    for param in model.stream2.parameters(): param.requires_grad = False

    opt_phase1 = optim.AdamW(model.fusion.parameters(), lr=Config.LR_FUSION, weight_decay=Config.WEIGHT_DECAY)

    for epoch in range(Config.EPOCHS_PHASE_1):
        t_loss = train_epoch(model, train_loader, criterion, opt_phase1, scaler, device)
        v_loss, v_metrics = validate_epoch(model, val_loader, criterion, device)

        logger.info(f"\n{'='*40}\n[PHASE 1] Epoch {epoch+1}/{Config.EPOCHS_PHASE_1}\n{'='*40}")
        logger.info(f"Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Acc: {v_metrics['acc']*100:.2f}%")
        logger.info(f"Precision: {v_metrics['precision']:.4f} | Recall: {v_metrics['recall']:.4f} | F1: {v_metrics['f1']:.4f} | ROC-AUC: {v_metrics['auc']:.4f}")
        logger.info(f"Confusion Matrix:\n{v_metrics['cm']}")

    # =========================================================================
    # PHASE 2: Differential Fine-Tuning
    # Unfreeze deep layers and train with ultra-low learning rate
    # =========================================================================
    logger.info("🌪️ PHASE 2: Unfreezing Backbones for Fine-Tuning...")

    # Unfreeze the top layers of EfficientNet and MobileNet
    for param in model.stream1.features[-3:].parameters(): param.requires_grad = True
    for param in model.stream2.features[-3:].parameters(): param.requires_grad = True

    # Differential Learning Rates (Fusion gets 10x higher LR than Backbones)
    opt_phase2 = optim.AdamW([
        {"params": filter(lambda p: p.requires_grad, model.stream1.parameters()), "lr": Config.LR_FINETUNE},
        {"params": filter(lambda p: p.requires_grad, model.stream2.parameters()), "lr": Config.LR_FINETUNE},
        {"params": model.fusion.parameters(), "lr": Config.LR_FINETUNE * 10}
    ], weight_decay=Config.WEIGHT_DECAY)

    # FIX: Removed `verbose=True` (Deprecated in PyTorch 2.2+, throws warnings)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_phase2, mode='min', factor=0.5, patience=2)

    start_time = time.time()
    for epoch in range(Config.EPOCHS_PHASE_2):
        t_loss = train_epoch(model, train_loader, criterion, opt_phase2, scaler, device)
        v_loss, v_metrics = validate_epoch(model, val_loader, criterion, device)

        scheduler.step(v_loss)

        logger.info(f"\n{'='*40}\n[PHASE 2] Epoch {epoch+1}/{Config.EPOCHS_PHASE_2}\n{'='*40}")
        logger.info(f"Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Acc: {v_metrics['acc']*100:.2f}%")
        logger.info(f"Precision: {v_metrics['precision']:.4f} | Recall: {v_metrics['recall']:.4f} | F1: {v_metrics['f1']:.4f} | ROC-AUC: {v_metrics['auc']:.4f}")
        logger.info(f"Confusion Matrix:\n{v_metrics['cm']}")

        # Save Best Model based on ROC-AUC (Standard for Imbalanced Forensic tasks)
        if v_metrics['auc'] > best_auc:
            best_auc = v_metrics['auc']
            patience_counter = 0

            torch.save({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': opt_phase2.state_dict(),
                'best_auc': best_auc,
            }, Config.BEST_MODEL_PATH)
            logger.info(f"   🌟 New Best AUC: {best_auc:.4f} -> Model Saved!")
        else:
            patience_counter += 1
            if patience_counter >= Config.PATIENCE:
                logger.info(f"🛑 Early Stopping triggered at epoch {epoch+1}")
                break

    total_time = (time.time() - start_time) / 60
    logger.info(f"✅ SOTA Training Complete in {total_time:.1f} minutes. Best Val AUC: {best_auc:.4f}")

if __name__ == "__main__":
    main()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
